# CTI-RCM walkthrough — CVE description → CWE

**Task:** read a CVE description and name the weakness type (CWE id).
**Scoring:** set F1 over CWE ids. **Cadence:** hourly. **Leak-resistance:** moderate.

This notebook walks the three stages — **dataflow → tasking → scoring** — over a couple of
examples using the project's real functions.

In [22]:
import os, sys, json

def _find_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.isdir(os.path.join(d, "src", "glokta")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("could not locate repo root (a dir containing src/glokta)")

ROOT = _find_root(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))

# Best-effort load of the repo .env so live HF calls have HF_TOKEN; no dotenv dependency.
_envp = os.path.join(ROOT, ".env")
if os.path.exists(_envp):
    for _line in open(_envp):
        _s = _line.strip()
        if _s and not _s.startswith("#") and "=" in _s:
            _k, _v = _s.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("TESTING", "1")  # relax settings validators if .env is absent

MODEL = "huggingface/meta-llama/Llama-3.1-8B-Instruct"
LIVE = bool(os.environ.get("HF_TOKEN"))
print("repo root :", ROOT)
print("model     :", MODEL)
print("LIVE calls:", LIVE, "(set HF_TOKEN to enable real inference)")

def run_model(prompt, canned, max_tokens=256):
    """Call the model live if HF_TOKEN is set, else return a canned example response."""
    if LIVE:
        from glokta.infrastructure.cti.inference import complete
        try:
            return complete(MODEL, prompt, max_tokens=max_tokens, timeout=60.0, max_retries=1)
        except Exception as exc:
            print("[live call failed -> canned]", type(exc).__name__, str(exc)[:80])
            return canned
    print("[offline -> canned response]")
    return canned

repo root : /Users/jake/Projects/glokta
model     : huggingface/meta-llama/Llama-3.1-8B-Instruct
LIVE calls: True (set HF_TOKEN to enable real inference)


## 1. Dataflow — raw CVE record → normalised RCM item
`normalise_cve_record` extracts the English description and the CWE label, applying CNA→ADP→NVD provenance.

In [23]:
# A CVE JSON 5.0 record (cvelistV5 shape). In production these come from the delta feed
# (fetch_recent_cve_records); here we use a representative in-memory example.
CVE_RECORD = {
    "cveMetadata": {"cveId": "CVE-2024-12345", "datePublished": "2024-05-01T10:00:00.000Z",
                    "dateUpdated": "2024-05-20T10:00:00.000Z", "state": "PUBLISHED"},
    "containers": {
        "cna": {
            "descriptions": [{"lang": "en",
                "value": "A SQL injection vulnerability in Acme Portal allows a remote "
                         "unauthenticated attacker to execute arbitrary SQL via the search parameter."}],
            "problemTypes": [{"descriptions": [{"lang": "en", "cweId": "CWE-89",
                              "description": "SQL Injection"}]}],
            "metrics": [{"cvssV3_1": {"vectorString": "CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H",
                                       "baseScore": 9.8}}],
        },
        # CISA-ADP (Vulnrichment) container — here it agrees with the CNA on the CWE.
        "adp": [{"providerMetadata": {"shortName": "CISA-ADP", "dateUpdated": "2024-05-20T10:00:00.000Z"},
                 "problemTypes": [{"descriptions": [{"lang": "en", "cweId": "CWE-89"}]}]}],
    },
}

In [24]:
from glokta.infrastructure.cti.connectors.cve import normalise_cve_record

items = normalise_cve_record(CVE_RECORD, source_revision="walkthrough")
rcm = next(i for i in items if i.task == "rcm")
print("external_id        :", rcm.external_id)
print("input_text         :", rcm.input_text[:90], "...")
print("label (ground truth):", rcm.label)
print("label_provenance   :", rcm.label_provenance)
print("authority_agreement:", rcm.authority_agreement)
print("input_date         :", rcm.input_date, "| label_date:", rcm.label_date)

external_id        : CVE-2024-12345
input_text         : A SQL injection vulnerability in Acme Portal allows a remote unauthenticated attacker to e ...
label (ground truth): {'cwe': ['CWE-89']}
label_provenance   : {'cna': ['CWE-89'], 'adp': ['CWE-89']}
authority_agreement: agree
input_date         : 2024-05-01 | label_date: 2024-05-01


## 2. Tasking — build the prompt and call the model
`build_prompt` renders the RCM template; `run_model` calls the model (live or canned).

In [25]:
from glokta.infrastructure.cti.prompts import build_prompt, parse_response

prompt = build_prompt("rcm", rcm.input_text)
print(prompt)
print("-" * 70)
response = run_model(prompt, canned="The flaw is SQL injection.\nAnswer: CWE-89")
print("model response:", repr(response))
print("parsed CWEs   :", parse_response("rcm", response))

You are a vulnerability analyst. Given the CVE description below, identify the most appropriate CWE (Common Weakness Enumeration) identifier(s).

CVE description:
A SQL injection vulnerability in Acme Portal allows a remote unauthenticated attacker to execute arbitrary SQL via the search parameter.

Respond with only the CWE id(s) in the form CWE-XXX.
Answer:
----------------------------------------------------------------------
model response: 'CWE-89'
parsed CWEs   : {'CWE-89'}


## 3. Scoring — compare prediction to the label
`evaluate_item` parses + scores in one step (set F1).

In [26]:
from glokta.infrastructure.cti.evaluator import evaluate_item

scored = evaluate_item("rcm", rcm.label, response)
print("score    :", scored.score)
print("correct  :", scored.correct)
print("breakdown:", scored.breakdown)
print("parsed   :", scored.parsed_output)

score    : 1.0
correct  : True
breakdown: {'precision': 1.0, 'recall': 1.0, 'tp': 1, 'fp': 0, 'fn': 0}
parsed   : {'cwe': ['CWE-89']}


## 4. A handful of examples
Score several model answers to see partial credit (set F1).

In [27]:
examples = {
    "exact":        "Answer: CWE-89",
    "extra guess":  "Could be CWE-89 or CWE-79\nAnswer: CWE-89, CWE-79",
    "wrong":        "Answer: CWE-22",
    "no answer":    "I cannot determine the weakness.",
}
for label, resp in examples.items():
    s = evaluate_item("rcm", rcm.label, resp)
    print(f"{label:12} score={s.score:.3f} correct={s.correct} pred={s.parsed_output['cwe']}")

exact        score=1.000 correct=True pred=['CWE-89']
extra guess  score=0.667 correct=False pred=['CWE-79', 'CWE-89']
wrong        score=0.000 correct=False pred=['CWE-22']
no answer    score=0.000 correct=False pred=[]


**Takeaway:** RCM is a clean objective task — exact CWE match scores 1.0, an extra wrong guess lowers precision (F1), a miss scores 0.0. Provenance (`authority_agreement`) records CNA/ADP/NVD agreement as a difficulty stratum.